# ETL — catalogo_fallas — Mantenimiento preventivo de bomba centrífuga

**Fuente:** snapshot `data/raw/catalogo_fallas_raw.csv` (extraído de PostgreSQL).

Este notebook procesa únicamente la tabla `catalogo_fallas`: carga el CSV crudo, aplica la transformación y limpieza y exporta el dataframe limpio a `data/processed/`.

## 1. Configuración

Imports, rutas del proyecto y carga de variables de entorno.

In [14]:
from pathlib import Path
import os
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# --- Rutas del proyecto ---
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / ".env").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_CATALOGO_FALLAS = PROJECT_ROOT / "data" / "raw" / "catalogo_fallas_raw.csv"
RAW_DATA_EQUIPOS = PROJECT_ROOT / "data" / "raw" / "equipos_raw.csv"
RAW_DATA_LECTURAS_SENSORES = PROJECT_ROOT / "data" / "raw" / "lecturas_sensores_raw.csv"
RAW_DATA_ORDENES_MANTENIMIENTO = PROJECT_ROOT / "data" / "raw" / "ordenes_mantenimiento_raw.csv"

PROCESSED_FILE = PROCESSED_DIR / "catalogo_fallas_clean.parquet"



## 2. Análisis, transformación y limpieza

### Tabla catalogo_fallas

In [15]:
df_catalogo_fallas = pd.read_csv(RAW_DATA_CATALOGO_FALLAS)
# df_a (data frame analisis - data set de analisis) copia del dataset para mantener una copia original intacta (df)
df_catalogo_fallas_w = df_catalogo_fallas.copy()

df_catalogo_fallas_w.head()

,codigo_falla,descripcion,severidad
0,F01,Desgaste severo en rodamientos,Alta
1,F02,Fuga en sello mecánico,Media
2,F03,Cavitación por obstrucción,Alta
3,F04,Sobrecalentamiento eléctrico,Media


In [16]:
df_catalogo_fallas_w.shape

(4, 3)

In [17]:
df_catalogo_fallas_w.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   codigo_falla  4 non-null      str  
 1   descripcion   4 non-null      str  
 2   severidad     4 non-null      str  
dtypes: str(3)
memory usage: 377.0 bytes


In [18]:
print('nulos: ', df_catalogo_fallas_w.isna().sum().sum())
print('duplicados: ', df_catalogo_fallas_w.duplicated().sum())

nulos:  0
duplicados:  0


In [19]:
df_catalogo_fallas_w.columns

Index(['codigo_falla', 'descripcion', 'severidad'], dtype='str')

In [20]:
print(df_catalogo_fallas_w['codigo_falla'].unique())
print(df_catalogo_fallas_w['descripcion'].unique())
print(df_catalogo_fallas_w['severidad'].unique())

<ArrowStringArray>
['F01', 'F02', 'F03', 'F04']
Length: 4, dtype: str
<ArrowStringArray>
['  Desgaste severo en rodamientos  ',          'Fuga en sello mecánico   ',
       '  Cavitación por obstrucción',       'Sobrecalentamiento eléctrico']
Length: 4, dtype: str
<ArrowStringArray>
['Alta', 'Media']
Length: 2, dtype: str


In [21]:
df_catalogo_fallas_w['descripcion'] = df_catalogo_fallas_w['descripcion'].str.strip()
print(df_catalogo_fallas_w['descripcion'].unique())

<ArrowStringArray>
['Desgaste severo en rodamientos',         'Fuga en sello mecánico',
     'Cavitación por obstrucción',   'Sobrecalentamiento eléctrico']
Length: 4, dtype: str


In [22]:
df_catalogo_fallas_w['descripcion'] = df_catalogo_fallas_w['descripcion'].replace({
    'Desgaste severo en rodamientos' : 'desgaste_severo_en_rodamientos',
    'Fuga en sello mecánico' : 'fuga_en_sello_mecanico',
    'Cavitación por obstrucción' : 'cavitacion_por_obstruccion',
    'Sobrecalentamiento eléctrico' : 'sobrecalentamiento_electrico'
})
print(df_catalogo_fallas_w['descripcion'].unique())

<ArrowStringArray>
['desgaste_severo_en_rodamientos',         'fuga_en_sello_mecanico',
     'cavitacion_por_obstruccion',   'sobrecalentamiento_electrico']
Length: 4, dtype: str


In [23]:
df_catalogo_fallas_w.head()

,codigo_falla,descripcion,severidad
0,F01,desgaste_severo_en_rodamientos,Alta
1,F02,fuga_en_sello_mecanico,Media
2,F03,cavitacion_por_obstruccion,Alta
3,F04,sobrecalentamiento_electrico,Media


In [24]:
print('Resumen del Dataset Work luego de transformaciones y limpieza')
print('=' * 61)
print('DF_Analysis: ', df_catalogo_fallas.shape, "-> DF_Work: ", df_catalogo_fallas_w.shape)
print('=' * 61)
print('Columnas antes: ', list(df_catalogo_fallas.columns))
print('Columnas después:', list(df_catalogo_fallas_w.columns))
print('=' * 61)
print('DF_Work tiene: ', df_catalogo_fallas_w.isna().sum().sum(), ' Valores Nulos')
print('DF_Work tiene: ', df_catalogo_fallas_w.duplicated().sum(), ' Valores Duplicados')
print('=' * 61)
print('Tipos de datos finales:')
print(df_catalogo_fallas_w.dtypes.to_string())
print('=' * 61)


Resumen del Dataset Work luego de transformaciones y limpieza
DF_Analysis:  (4, 3) -> DF_Work:  (4, 3)
Columnas antes:  ['codigo_falla', 'descripcion', 'severidad']
Columnas después: ['codigo_falla', 'descripcion', 'severidad']
DF_Work tiene:  0  Valores Nulos
DF_Work tiene:  0  Valores Duplicados
Tipos de datos finales:
codigo_falla    str
descripcion     str
severidad       str


In [25]:
df_catalogo_fallas_w.head()

,codigo_falla,descripcion,severidad
0,F01,desgaste_severo_en_rodamientos,Alta
1,F02,fuga_en_sello_mecanico,Media
2,F03,cavitacion_por_obstruccion,Alta
3,F04,sobrecalentamiento_electrico,Media


## 3. Exportación a Parquet

Se guarda el dataframe limpio `df_catalogo_fallas_w` en `data/processed/catalogo_fallas_clean.parquet`.

In [26]:
# Exportación del dataframe limpio a Parquet
archivo_temporal = PROCESSED_FILE.with_suffix(".parquet.tmp")

df_catalogo_fallas_w.to_parquet(archivo_temporal, index=False)
archivo_temporal.replace(PROCESSED_FILE)

print(f"Filas: {len(df_catalogo_fallas_w):,} | Columnas: {df_catalogo_fallas_w.shape[1]}")

Filas: 4 | Columnas: 3
